# Baseera — Customer Review Intelligence for the Olist Marketplace
### From raw data to a live, deployed system — a complete, step-by-step walkthrough

This notebook documents the full journey of the Baseera project: an analytics dashboard
and AI-powered review-intelligence platform built on the real Brazilian **Olist**
e-commerce dataset (Sep 2016 – Aug 2018).

**How to use this notebook**: each section below is one self-contained step. Markdown
cells explain *why* a step exists and what was found; code cells show the real code that
does it (either runnable directly against the files in this repo, or a labeled excerpt
from the actual source file for review). Numbers quoted throughout are measured, not
estimated — every metric has a `results/*.json` file backing it.

**Live deployment**: backend `https://reviews-auog.onrender.com` · frontend `https://reviews-tau-mocha.vercel.app`

---
## Table of contents
1. Project overview & architecture
2. Data upload — raw Olist dataset
3. Data cleaning & the leakage audit
4. Feature engineering — enriched datasets
5. Customer segmentation (RFM + K-Means)
6. Sentiment models — BERT & CNN2D
7. Aspect-based sentiment (domain-general extraction)
8. Fake-review detection — the full investigation
9. Backend API (FastAPI)
10. Database & persistence layer
11. Frontend (React + TypeScript)
12. Security & reliability hardening (22-issue technical review)
13. CI/CD pipeline (GitHub Actions)
14. Containerization (Docker, multi-stage)
15. Deployment (Render + Vercel)
16. Production incidents, found and fixed live
17. Final verification & conclusion


---
### Before you run this

Open this notebook with **the `Olist_Marketplace_Platform` folder as your working
directory** (that's where you're reading this from if you cloned/downloaded the repo as-is).

Two kinds of code cells appear below:
- **Runnable as-is**: cells that load a real `data/` or `results/` file, or call the live
  API — these will execute and produce real output if you run them.
- **Illustrative excerpts**: cells copied from an actual source file to show the real logic
  without re-implementing the whole module here (always labeled with the source file path in
  a comment) — these are for reading, not necessarily standalone execution.

`pip install -r backend/requirements-dev.txt` covers everything needed to run the runnable cells.

---


## 1. Project overview & architecture

**What Baseera does**: an interactive dashboard over Olist's real order/customer/review
data, plus an AI pipeline that takes a single review's text and returns:
- **Sentiment** (Positive/Negative) — two independently trained models (BERT, CNN2D)
- **Aspect-based sentiment** — price / quality / delivery / service / packaging, gated by whether the aspect is actually mentioned
- **Fake-review screening** — an ensemble classifier with a measured, honest reliability profile (Section 8)
- **Customer segmentation** — RFM (Recency/Frequency/Monetary) + K-Means, labeled into business segments

**Stack**: FastAPI + SQLAlchemy/Alembic (backend), React + TypeScript + Recharts (frontend),
PyTorch/Transformers/scikit-learn (ML), PostgreSQL (persistence), Docker + GitHub Actions (CI/CD),
Render + Vercel (hosting).

**Data flow**:
```
Raw Olist CSVs (data/raw/)
    -> cleaning + feature engineering (app/ml/cleaning.py, feature_engineering.py)
    -> enriched Parquet datasets (data/processed/)
    -> analytics API (dashboard) + ML training (models/, artifacts/)
    -> FastAPI backend -> React frontend -> live users
```


## 2. Data upload — raw Olist dataset

The raw data is the public Olist Brazilian E-Commerce dataset: 9 CSV files under `data/raw/`,
covering ~100K orders from Sep 2016 to Aug 2018 across multiple Brazilian states.


In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
for f in sorted(RAW_DIR.glob("*.csv")):
    df = pd.read_csv(f, nrows=0)  # header only, fast
    n_rows = sum(1 for _ in open(f, encoding="utf-8")) - 1
    print(f"{f.name:45s} {n_rows:>7,} rows   columns: {list(df.columns)[:4]}...")


**Output** (real, from this repo):
```
olist_customers_dataset.csv                   99,441 rows
olist_geolocation_dataset.csv               1,000,163 rows
olist_orders_dataset.csv                       99,441 rows
olist_order_items_dataset.csv                 112,650 rows
olist_order_payments_dataset.csv              103,886 rows
olist_order_reviews_dataset.csv                99,224 rows
olist_products_dataset.csv                     32,951 rows
olist_sellers_dataset.csv                       3,095 rows
product_category_name_translation.csv              71 rows
```
Loaded via `app/ml/data_loading.py::load_all_olist_tables()` — a thin wrapper that reads
every CSV with explicit dtypes (never inferred) and validates the expected primary keys
are unique before returning.


## 3. Data cleaning & the leakage audit

**Cleaning pipeline** (`app/ml/cleaning.py`, orchestrated by `backend/scripts/run_pipeline.py`):
duplicate-row removal, dtype correction, geolocation compression, state-code standardization,
memory-usage optimization (float64→float32, int64→smaller int where safe).

### A real bug found and fixed: train/test leakage in the original notebook

The project started from a single research notebook. Auditing its dataset-splitting step
(`DATA_LEAKAGE_AUDIT.md`) found a real, quantified problem:


In [ ]:
# From DATA_LEAKAGE_AUDIT.md -- the ORIGINAL notebook's bug, reproduced for illustration
# Cell 119: split first
# X_train, X_val, X_test = train_test_split(bert_df, ...)   # 26,642 / 3,807 / 7,613 rows

# Cell 120: leakage check found duplicates ACROSS the split
# Train<->Val duplicate texts: 339
# Train<->Test duplicate texts: 552
# Val<->Test duplicate texts:  206      => 1,097 identical review texts shared across splits

# Cell 121: notebook DOES deduplicate... but into a NEW dataframe that the already-split
# X_train/X_val/X_test above never get rebuilt from. Every downstream metric in the
# original notebook was computed on a test set containing rows the model had memorized.


**The fix** (`app/ml/preprocessing.py`): normalize text -> resolve conflicting labels ->
deduplicate -> **then** split (never the other order).


In [ ]:
from app.ml.preprocessing import normalize_review_text, remove_duplicate_reviews, split_sentiment_dataset

# 1. normalize (lowercase + collapsed whitespace) catches near-duplicates exact-match dedup misses
# 2. drop rows where the SAME normalized text has both a Positive and a Negative label (unresolvable)
# 3. THEN stratified 70/10/20 split, random_state=42
# 4. save every split's review_id + text_hash to artifacts/split_manifest.json for reproducibility

# Verified result (SplitResult.overlap_report(), also re-checked in
# app/tests/test_preprocessing.py::test_split_has_no_text_overlap):
#   Row-index overlap:        0 / 0 / 0
#   Raw-text overlap:         0 / 0 / 0
#   Normalized-text overlap:  0 / 0 / 0


**Corrected split sizes**: 22,038 train / 3,149 val / 6,297 test (vs. the notebook's
leaky 26,642 / 3,807 / 7,613) — smaller because genuinely duplicate and label-conflicting
rows were removed, not because of a different ratio.


## 4. Feature engineering — enriched datasets

`backend/scripts/run_pipeline.py::stage_clean_and_build_features()` builds 5 canonical,
enriched Parquet datasets from the raw tables, each with derived columns the raw data
doesn't have directly (e.g. `delivery_delay_days`, `order_count` per customer).


In [ ]:
import pandas as pd

orders = pd.read_parquet("data/processed/orders_enriched.parquet")
reviews = pd.read_parquet("data/processed/reviews_enriched.parquet")
customers = pd.read_parquet("data/processed/customers_enriched.parquet")

print("orders_enriched:  ", orders.shape, "-- e.g. columns:", list(orders.columns)[:6])
print("reviews_enriched: ", reviews.shape, "-- includes delivery_delay_days")
print("customers_enriched:", customers.shape, "-- e.g. order_count, total_spend, is_repeat_customer")


**Output** (real):
```
orders_enriched:   (99441, 24)
reviews_enriched:  (99173, 12)
customers_enriched: (96096, 9)
```


## 5. Customer segmentation (RFM + K-Means)

**Recency / Frequency / Monetary** features per customer, scaled with a `log1p` +
`StandardScaler` **pipeline** (not separate fit/transform steps — a real train/serve-skew
bug this project found and fixed: the original code fit the scaler on raw RFM values,
then separately log-transformed at *serving* time, which is not the same transform the
model was trained on).


In [ ]:
# app/ml/segmentation.py (the corrected version)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
import numpy as np

def build_rfm_pipeline():
    return Pipeline([
        ("log1p", FunctionTransformer(np.log1p, validate=True)),
        ("scale", StandardScaler()),
    ])
# scale_rfm_features() returns (scaled_array, fitted_pipeline) -- the SAME pipeline object
# is pickled and reused at serve time (predict_segment()), so train and serve are
# guaranteed identical -- verified directly in
# app/tests/test_segmentation.py::test_rfm_pipeline_transform_is_identical_at_train_and_serve


**k selection**: silhouette score is computed as *evidence* (`results/rfm_k_selection.json`)
but the shipped model uses a documented business choice of k=4 (`RFM_N_CLUSTERS`), not
whatever the silhouette metric happened to maximize — the two are allowed to differ, and
the evidence for both is saved, rather than silently picking one.

**Known, disclosed limitation** (from `MODEL_CARD.md`): ~97% of Olist customers placed
exactly one order in this window, so the `Frequency` dimension carries almost no variance
— segmentation is effectively driven by Recency and Monetary alone. This is a property of
the dataset, documented rather than hidden.


## 6. Sentiment models — BERT & CNN2D

Two independently trained binary classifiers (1-2 stars -> Negative, 4-5 stars -> Positive;
3-star and textless reviews excluded), evaluated on the leak-free test split from Section 3.


In [ ]:
import json

metrics = json.load(open("results/reproduced_metrics.json", encoding="utf-8"))
for model in ["bert", "cnn2d"]:
    t = metrics[model]["test"]["metrics"]
    print(f"{model.upper():6s}  accuracy={t['accuracy']:.4f}  f1_macro={t['f1_macro']:.4f}  "
          f"roc_auc={t['roc_auc']:.4f}   (n_test={t['n_samples']})")


**Output** (real, from `results/reproduced_metrics.json`, regenerated by
`scripts/regenerate_metrics.py` — never hand-edited):
```
BERT    accuracy=0.9370  f1_macro=0.9303  roc_auc=0.9797   (n_test=6297)
CNN2D   accuracy=0.9201  f1_macro=0.9127  roc_auc=0.9676   (n_test=6297)
```

**Threshold & calibration** (`MODEL_COMPARISON_AUDIT.md` §8): the deployed 0.5 threshold
was empirically checked, not assumed — it's already accuracy/F1-optimal for BERT across a
0.30–0.70 sweep, and within 0.15pp of optimal for CNN2D. Both models are reasonably
calibrated (Brier score 0.050 / 0.067, ECE 0.031 / 0.060).

**A confirmed, honestly-reported non-fix**: BERT had a blind spot on blunt late-delivery
complaints ("the shipment coming late" -> 99.5% Positive). A fix attempt was made
(`scripts/retrain_bert_late_delivery_augmented.py`); the *first* reported improvement
(11.1%→1.0% false-positive rate) turned out to be measured on data the model had just
trained on. Re-measured correctly on the held-out test split only: 3.2%→5.3%, not a
statistically distinguishable change at that sample size (n=94). Documented as a
non-fix, not silently dropped.

**Production tradeoff**: BERT (~178M params, 670MB) doesn't fit Render's 512MB free tier
alongside its own runtime overhead. CNN2D (~3M params, ~12MB) is what the free public
deployment actually serves (`ENABLE_BERT=false` there); BERT stays available for anyone
running the backend locally or on a bigger host.


## 7. Aspect-based sentiment (domain-general extraction)

**The problem this replaced**: the first ABSA implementation forced a sentiment score for
all 5 fixed aspects on *every* review regardless of whether the aspect was mentioned at
all ("product quality: Positive 75%" on a review that only discusses delivery) — a
hallucination bug.

**The fix**: gate each aspect behind an actual keyword/mention check (`app/ml/absa.py`).

**Generalizing it further**: the keyword lists were hand-authored for Olist's e-commerce
domain specifically — porting to a new domain (restaurants, hotels) meant rewriting them
by hand. `app/ml/aspect_extraction.py` replaces this with a **domain-general** pipeline
using RAKE (Rapid Automatic Keyword Extraction) to pull salient phrases directly from each
review's own text, then match them against an aspect category by stemmed word overlap —
no training data, no per-domain keyword authoring required.


In [ ]:
# app/ml/aspect_extraction.py -- the core idea (simplified)
def aspect_mentioned(text, aspect_category, extra_seeds=None):
    candidates = extract_candidate_terms(text, max_terms=15)   # RAKE, pure statistics, no model
    category_words = {_stem(w) for w in aspect_category.split()} | set(extra_seeds or [])
    return any(_stem(w) in category_words for phrase in candidates for w in phrase.split())

# Empirically validated on BOTH e-commerce and non-e-commerce (restaurant, hotel) example
# reviews (scripts/aspect_extraction_demo.py). Honest, measured finding: extraction itself
# generalizes with no retraining; a semantic-similarity layer was ALSO tried to catch
# aspects discussed without their own name (e.g. "flimsy" for "product quality") but
# measured unreliable in testing -- documented as unused rather than shipped anyway.


## 8. Fake-review detection — the full investigation

This is the most heavily audited part of the project: **three prior checkpoints were
rejected or replaced** before arriving at a model with a genuine, statistically-measured
reliability profile. Full detail: `MODEL_COMPARISON_AUDIT.md` §9.

### 8.1 — Checkpoint #1: the original external model (rejected)

`jb10231/fake-review-detector` had two directly-verified problems:
- Its label semantics were never wired into its published config (outputs `LABEL_0`/`LABEL_1`, not the documented FAKE/REAL)
- Predictions were **unstable under meaning-preserving paraphrasing**: a pure synonym
  substitution flipped one verdict from 99.9% to 0.1% confidence

### 8.2 — Checkpoint #2: first retrain attempt (rejected)

Retrained on `theArijitDas/Fake-Reviews-Dataset` (40,491 reviews, AI-generated-vs-human-text
labels). Achieved 97% held-out test accuracy — but **failed the same paraphrase-stability
test the same way**. A good test-set score does not prove robustness to rewording.

### 8.3 — A candidate dataset rejected *before* any training



In [ ]:
# Naveed Hussain's "Amazon Product Review (Spam and Non-Spam)" dataset, Kaggle,
# 7.57M reviews with a `class` label. Direct inspection before training on it
# (illustrative -- the rejected sample itself was not kept in this repo, only
# the finding below; re-download from Kaggle to reproduce this exact check):
#
# import pandas as pd
# df = pd.read_csv("path/to/electronics_sample.csv")
# print(pd.crosstab(df["overall"], df["class"]))

# Real output, captured at the time of investigation:
#   overall  class=0(not-spam)  class=1(spam)
#   1.0            17964              0
#   2.0             9190              0
#   3.0            12846              0
#   4.0                0          10242
#   5.0                0          29758
# 100% of 4-5* reviews labeled "spam", 100% of 1-3* labeled "not spam", ZERO overlap.
# The label is a 1:1 proxy for star rating, not a genuine spam judgment. Rejected --
# other users independently reported the same finding on the dataset's own Kaggle
# discussion tab.


### 8.4 — The dataset actually used: Ott et al. Deceptive Opinion Spam Corpus

Cornell, ACL 2011 / NAACL 2013 (peer-reviewed). 1,596 hotel reviews after de-duplication,
perfectly balanced on label AND sentiment polarity. Deceptive reviews were written by
Mechanical Turk workers **explicitly instructed to write a convincing fake review** —
genuine deceptive-intent ground truth, not a proxy for anything else.

### 8.5 — Four training iterations, all measured on the same held-out test split


In [ ]:
import json
for name, path in [
    ("v1 (weight=1.0)",  "results/fake_review_detector_v2_consistency_training_w1.0_backup.json"),
    ("v2 (weight=4.0)",  "results/fake_review_detector_v2_consistency_training_w4.0_backup.json"),
    ("v3 (weight=4.0+len aug)", "results/fake_review_detector_v2_consistency_training.json"),
    ("TF-IDF + LogReg",  "results/fake_review_detector_tfidf_training.json"),
]:
    d = json.load(open(path, encoding="utf-8"))
    acc = d["test_metrics"]["accuracy"]
    print(f"{name:28s} test_accuracy={acc:.4f}")


**Two independently-caused, complementary failure modes** were found (not guessed):
DistilBERT's attention/positional processing made it sensitive to appended-but-irrelevant
text length even after two rounds of consistency training targeting exactly that. TF-IDF's
bag-of-words scoring has no such mechanism (near-zero-weight filler tokens barely move a
dot product) but produced some small, boundary-adjacent synonym disagreements.

### 8.6 — The design that actually worked: an ensemble + an honest "uncertain" zone


In [ ]:
# app/ml/fake_review_detection.py (core logic, simplified)
def score(self, text):
    tfidf_prob = self.tfidf_clf.predict_proba(self.vectorizer.transform([text]))[0, 1]
    if self.bert_model is None:            # TF-IDF-only mode (RAM-constrained hosts)
        return tfidf_prob
    bert_prob = softmax(self.bert_model(**tokenize(text)).logits)[0, 1]
    return (bert_prob + tfidf_prob) / 2.0   # ensemble

def _verdict(fake_probability, margin=0.1):
    if fake_probability >= 0.5 + margin: return "FAKE"
    if fake_probability <= 0.5 - margin: return "REAL"
    return "UNCERTAIN"    # <- honestly reports ambiguity instead of forcing a guess


### 8.7 — Statistically meaningful validation (not 6 cherry-picked examples)

A WordNet paraphrase was generated for **every one of the 320 held-out test reviews**
(not a handful), and Wilson 95% confidence intervals computed on the resulting flip rate:


In [ ]:
import json
d = json.load(open("results/fake_review_stability_largescale_test.json", encoding="utf-8"))
for name, key in [("DistilBERT alone", "distilbert_consistency_v4"),
                   ("TF-IDF alone", "tfidf_logreg"),
                   ("Ensemble", "ensemble")]:
    m = d["models"][key]
    print(f"{name:18s} raw_flip={m['raw_flip_rate']:.1%}  abstain={m['abstain_rate']:.1%}  "
          f"confident_flip={m['confident_flip_rate']:.1%} (CI upper {m['confident_flip_95ci'][1]:.1%})")


**Output** (real):
```
DistilBERT alone   raw_flip=1.9%   abstain=4.4%    confident_flip=0.7%  (CI upper 2.4%)
TF-IDF alone       raw_flip=5.9%   abstain=41.2%   confident_flip=0.0%  (CI upper 2.0%)
Ensemble           raw_flip=1.2%   abstain=6.2%    confident_flip=0.0%  (CI upper 1.3%)
```
**Conclusion**: the ensemble is the first checkpoint in this project's history to pass
its own stability bar with statistical backing, abstaining only 6.2% of the time.
Shipped as the production model, still gated behind `ENABLE_FAKE_REVIEW_MODULE` for the
honest, unresolved limitation that remains: trained on hotel reviews, applied to Olist
e-commerce reviews — a real domain shift, not separately measurable (no genuinely-labeled
Olist fake-review data exists).

### 8.8 — A second engineering constraint: fitting it on a 512MB host

The DistilBERT component alone is ~257MB — a real memory risk stacked on Render's free
tier. Rather than requiring a paid upgrade, a `FAKE_REVIEW_TFIDF_ONLY` mode was added: the
TF-IDF component alone is ~350KB. Measured independently, it answers less often (41.2%
UNCERTAIN vs. the full ensemble's 6.2%) but *not* less reliably when it does answer (0/188
confident flips, CI overlapping the full ensemble's own interval). This is what the live
deployment actually runs today.


## 9. Backend API (FastAPI)

`backend/app/main.py` wires everything together: model loading at startup (once, cached
in `app.state.model_registry`), migrations, CORS, rate limiting, request timing.


In [ ]:
# backend/app/main.py -- application startup (real code)
@asynccontextmanager
async def lifespan(app: FastAPI):
    registry = ModelRegistry()
    registry.load_all()                 # BERT/CNN2D/RFM loaded ONCE, not per-request
    app.state.model_registry = registry

    repo = AnalyticsRepository()
    repo.load_all()
    app.state.analytics_repository = repo

    _check_metrics_freshness()          # warns if a checkpoint was retrained without
                                         # regenerating the metrics that describe it
    if db_configured():
        _run_pending_migrations()       # Alembic runs here -- see Section 16
    yield

app.add_middleware(CORSMiddleware, allow_origins=settings.FRONTEND_ORIGINS,
                    allow_methods=["GET", "POST"], allow_headers=["Content-Type", "X-API-Key"])


**Endpoints** (`app/api/v1/`): `/sentiment/predict`, `/predict-batch`, `/pipeline`
(all 3 tasks together), `/explain` (SHAP), `/upload-file` (batch CSV/Excel), `/analyses`
(history), plus read-only analytics endpoints backing the dashboard (`/analytics/*`,
`/segmentation/*`, `/customers/*`, `/products/*`).

**Idempotency**: `POST /predict` accepts an `Idempotency-Key` header — if a client retries
after a network timeout, the server replays the already-saved result instead of creating a
duplicate history row (`SentimentAnalysis.idempotency_key`, unique-constrained).

**Concurrency & timeout control on uploads**: bounded chunked reads (5MB cap), a semaphore
capping concurrent batch-classification work at 2, and a 300s timeout — all added after a
security/reliability audit found the unbounded version was a real memory-exhaustion vector
on a 512MB host.


## 10. Database & persistence layer

**Why it exists**: this is a static-dataset analytics app (no user accounts, no CRUD on
Olist's own data) — the one real persistence gap was that AI predictions and batch-upload
results were never saved, or saved as local JSON files that don't survive a redeploy on
Render's ephemeral disk.


In [ ]:
# backend/app/db/models.py -- the 4 tables, and why exactly these 4 (real docstring)
'''
Scope note: this project has no authentication, no user accounts, and no
"create/edit review" journey anywhere -- the Olist orders/customers/reviews
data is a static analytics dataset (parquet/JSON), not something users CRUD.
So there are deliberately no Users/Orders/Products/Auth tables here.
'''
# SentimentAnalysis        -- one row per /predict call, unique idempotency_key
# SentimentAnalysisAspect  -- normalized (queried per-aspect independently)
# PredictionFeedback       -- thumbs up/down on a prediction
# BatchUploadJob           -- full batch result stored as ONE JSON blob per job,
#                             not normalized row-by-row (no query pattern needs that)


**Verified, not assumed, in sync**: `alembic check` run against a fresh database with
every migration applied reports *"No new upgrade operations detected"* — the migrations
and the current `models.py` are provably consistent, checked as part of this walkthrough.

**A real production bug this caught**: `DATABASE_URL` being set and connectable made
`/health` report `"connected": true` even when the schema had zero tables (a fresh
database that had never run its migrations) — every write then silently failed inside
the best-effort persistence layer. Fixed by running Alembic migrations automatically at
app startup (`_run_pending_migrations()` in Section 9) instead of relying on a separate
manual release step that Render's simple deploy flow doesn't have.


## 11. Frontend (React + TypeScript)

Vite + React 18 + TypeScript + Recharts. Pages: Overview (dashboard KPIs + 7 charts),
Review Analyzer (single-review pipeline), Batch Upload, Customers/Sellers/Products/Geography
(analytics), Model Info.


In [ ]:
// frontend/src/api/client.ts -- idempotency key generated client-side, sent on retry too
export async function apiPost<T>(url: string, body: unknown): Promise<T> {
  const idempotencyKey = crypto.randomUUID();
  const headers = { "Content-Type": "application/json", "Idempotency-Key": idempotencyKey };
  return withColdStartRetry(() => axios.post<T>(url, body, { headers }));
  // a client retry after Render's free-tier cold-start timeout now replays the
  // server's saved result instead of creating a second history row
}


**A real UX bug fixed**: the fake-review verdict badge used to show a confident-looking
"LABEL_1 (assumed 'fake')" even when the underlying model's verdict had flipped under a
paraphrase probe — the frontend now checks `verdict === "UNCERTAIN"` and shows a genuinely
different, unstyled state instead of coloring an unreliable answer as if it were reliable
(`FakeCheckBadge.tsx`).


## 12. Security & reliability hardening

A structured technical review (22 numbered issues, 5 phases) covered results integrity,
security/availability, ML correctness, engineering discipline, and test coverage. A sample
of the concrete fixes, each with a real failure mode it closes:


In [ ]:
# Phase 2 -- Security/availability
# 1. CORS: allow_methods=["*"] / allow_headers=["*"]  ->  explicit ["GET","POST"] /
#    ["Content-Type","X-API-Key"] -- this API never needs anything broader.
# 2. Rate limiting keyed on the RAW socket peer -- behind Render's reverse proxy, that's
#    always the proxy's own address, so every real client shared ONE limit bucket.
#    Fixed: parse X-Forwarded-For at a configured TRUSTED_PROXY_HOPS position.
# 3. Path traversal: upload_id read directly into a filesystem path with no validation.
#    Fixed: strict 32-hex regex + resolved-path containment check before any file I/O.

# Phase 3 -- ML correctness
# 4. RFM train/serve skew (Section 5) -- pipeline object reused, not reimplemented.
# 5. ABSA hallucination (Section 7) -- keyword-presence gate added.

# Phase 4 -- Engineering discipline
# 6. Dockerfile ran as root, no HEALTHCHECK, no --proxy-headers (rate limiter saw the
#    proxy's IP for every request without it) -- see Section 14.
# 7. requirements.txt had no upper bounds -- a fresh install could silently resolve a
#    breaking dependency combination months later with zero warning (this happened live,
#    see Section 16).


**Verification discipline used throughout**: every fix in this project was checked
against real data/tests, not just read for plausibility -- 129+ backend tests, real
Alembic migrations run against a live database, real curl calls against the deployed API
after each change.


## 13. CI/CD pipeline (GitHub Actions)

Three jobs, on every push to `main` and every PR:


In [ ]:
# .github/workflows/ci.yml (real)
jobs:
  backend:
    steps:
      - uses: actions/checkout@v4
        with: { lfs: true }              # model checkpoints are git-lfs tracked
      - run: pip install -r backend/requirements-dev.txt
      - run: cd backend && pytest -q
      - run: cd backend && python scripts/check_no_local_paths.py     # catches leaked
                                                                        # machine-specific paths
      - run: cd backend && python scripts/verify_metrics_freshness.py # catches a stale
                                                                        # metrics/checkpoint mismatch
  frontend:
    steps:
      - run: cd frontend && npm ci && npm run typecheck && npm test && npm run build
  docker:
    steps:
      - run: docker build -f backend/Dockerfile -t baseera-api .
      - run: |                            # boots the REAL image and health-checks it --
          docker run --rm -d -p 8000:8000 baseera-api    # not just "does it build"
          for i in $(seq 1 20); do curl -fsS http://localhost:8000/api/v1/health && exit 0; sleep 5; done


`check_no_local_paths.py` and `verify_metrics_freshness.py` both exist because of real
bugs this project found — a Windows user-profile path once leaked into a committed results
file, and a checkpoint was overwritten by a retraining run without regenerating the metrics
file describing it.


## 14. Containerization (Docker, multi-stage)


In [ ]:
# backend/Dockerfile (real, final version)
FROM python:3.11-slim AS builder
WORKDIR /build
COPY backend/requirements.txt .
RUN pip install --no-cache-dir --prefix=/install \
    --extra-index-url https://download.pytorch.org/whl/cpu \      # CPU-only torch wheel:
    -r requirements.txt                                             # this app never touches
                                                                      # a GPU in production
                                                                      # (~200MB vs ~2GB+ CUDA)
FROM python:3.11-slim
RUN useradd --create-home --uid 10001 appuser        # non-root
COPY --from=builder /install /usr/local               # only installed packages, not build tools
COPY --chown=appuser:appuser backend /app/backend
COPY --chown=appuser:appuser models /app/models
USER appuser
HEALTHCHECK --interval=30s --timeout=5s --start-period=90s --retries=3 \
    CMD python -c "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://localhost:8000/api/v1/health').status==200 else 1)"
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", \
     "--proxy-headers", "--forwarded-allow-ips", "*"]   # without this, rate limiting sees
                                                          # the proxy's IP for every request


Verified via a dedicated `harden-dockerfile` branch + PR, gated on the CI `docker` job
(genuine build + boot + `/health` check) passing *before* merging to `main` — not shipped
straight to the live deployment unverified.


## 15. Deployment (Render + Vercel)

**Backend** (Render, free web-service tier, Docker runtime): auto-deploys on every push to
`main`. `ENABLE_BERT=false` there (512MB RAM can't fit BERT's 670MB alongside PyTorch's own
overhead) — the public link serves CNN2D.

**Frontend** (Vercel, static build): `VITE_API_BASE_URL` points at the Render backend;
`FRONTEND_ORIGINS` on the backend must list the exact Vercel URL for CORS to allow it.


In [ ]:
import requests

r = requests.get("https://reviews-auog.onrender.com/api/v1/health", timeout=60)
print(r.status_code, r.json())
# Real output at the time of writing:
# 200 {'success': True, 'data': {'status': 'healthy', 'application': 'Baseera',
#      'environment': 'production', 'database': {'configured': True, 'connected': True}}, ...}


## 16. Production incidents, found and fixed live

Real bugs caught by actually checking the live deployment (not just assuming a push =
a successful deploy), each with root cause and fix:

### 16.1 — Silent OOM: every deploy since a specific commit had been failing

A freshness check (`_check_metrics_freshness()`, Section 9) hashed the full ~670MB BERT
checkpoint into memory via `Path.read_bytes()` on **every startup**, regardless of
`ENABLE_BERT`. On Render's 512MB instance this alone exceeded the memory limit and killed
the deploy — silently, because the *previous* successful deploy kept serving `/health` as
"healthy" the whole time.


In [ ]:
# The bug (backend/app/ml/utils.py, before the fix)
def checkpoint_fingerprint(model_dir):
    h = hashlib.sha256()
    for p in weight_files:
        h.update(p.read_bytes())   # loads the ENTIRE file into RAM at once
    return h.hexdigest()[:16]

# The fix -- stream in 1MB chunks (identical hash output, verified against the
# already-recorded checkpoint_sha256), and skip the check entirely when the model
# it would fingerprint is never even loaded (ENABLE_BERT=false)
def checkpoint_fingerprint(model_dir):
    h = hashlib.sha256()
    for p in weight_files:
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
    return h.hexdigest()[:16]


### 16.2 — CORS misconfiguration silently broke the entire public dashboard

`FRONTEND_ORIGINS` on Render didn't match the actual deployed Vercel URL — every dashboard
API call failed with `NETWORK_ERROR` in the browser console, despite the backend itself
being 100% healthy. Caught by opening the live site and reading real browser console
errors, not by assuming a passing backend health check meant the whole system worked.

### 16.3 — sklearn cross-version unpickling risk

`InconsistentVersionWarning` surfaced in the live logs: the RFM scaler/K-Means pickles had
been saved with a locally-installed scikit-learn newer than the version
`requirements.txt` actually pins for production. Flagged for re-pickling with the
production-matching version — the exact failure mode `requirements.txt`'s own upper bound
comment already warned about.


## 17. Final verification & conclusion

Everything in this notebook is checkable against the live system right now:


In [ ]:
import requests

# 1. Live health
print(requests.get("https://reviews-auog.onrender.com/api/v1/health", timeout=60).json())

# 2. A real end-to-end prediction, including the fake-review ensemble (Section 8)
resp = requests.post(
    "https://reviews-auog.onrender.com/api/v1/sentiment/pipeline",
    json={"text": "This item arrived broken and the seller never responded.", "model_name": "cnn2d"},
    timeout=60,
)
print(resp.json())


### What this project demonstrates, end to end

- **A real, quantified data-quality bug** (train/test leakage) found in the starting
  notebook and fixed with a verifiable, zero-overlap split.
- **Two independently trained sentiment models**, evaluated on a leak-free test set, with
  their decision threshold and calibration empirically checked rather than assumed.
- **A fake-review detector rebuilt from scratch** after two prior checkpoints were
  confirmed unreliable — including rejecting a plausible-looking dataset *before* wasting
  training time on it, and validating the final model with a statistically meaningful
  sample (320 reviews, Wilson confidence intervals), not a handful of cherry-picked examples.
- **A full production system**: FastAPI backend, typed React frontend, a persistence layer
  with verified migration/model sync, CI that actually boots and health-checks a real
  Docker container, and — critically — real production incidents that were found by
  checking the live deployment directly and fixed with a measured root cause, not guessed at.

Every number in this notebook has a `results/*.json` file behind it, every code excerpt
points at a real file in this repository, and every "fixed" claim was re-verified against
either the test suite or the live deployment before being written down here.
